<a href="https://colab.research.google.com/github/lehanvats/Harry-Potter-Classifier/blob/main/diltilBERT_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import pandas as pd
from transformers import pipeline

# Updated path for Google Colab
file_path = '/content/personality_classification_dataset.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

# Clean whitespace and split into lines
text_clean = re.sub(r'\\s*', '', text)
lines = text_clean.strip().split('\n')

data = []
for line in lines:
    parts = line.split('|')
    if len(parts) == 2:
        text_val = parts[0].strip()
        label_val = int(parts[1].strip())
        data.append({'text': text_val, 'label': label_val})

df = pd.DataFrame(data)
print(f"Loaded {len(df)} examples")
display(df.head(5))

Loaded 200 examples


,text,label
0,I once told my manager directly in a team meet...,0
1,When I see someone being treated unfairly at w...,0
2,"I've been told I'm too blunt, but I genuinely ...",0
3,Last quarter I jumped into a cross-team projec...,0
4,My biggest flaw is probably that I act on inst...,0


## Run for Zero Shot Trial

In [3]:
distilBERT = pipeline("zero-shot-classification", model ="typeform/distilbert-base-uncased-mnli")

#archetypes generated using AI
hogwarts_house = [
    "courageous, reckless, principled, confronts authority",        # 0: Gryffindor
    "loyal, hardworking, fair, avoids conflict, team player",       # 1: Hufflepuff
    "intellectual, curious, analytical, independent thinker",       # 2: Ravenclaw
    "ambitious, strategic, calculating, resourceful, networking"    # 3: Slytherin
]

sample_input_difficult = "I once quit a job on the spot because my boss asked me to sign off on a report which I knew was misleading. Some things are non-negotiable for me"

print(f"sample input complex: '{sample_input_difficult}'")
print(f"actual label should be: Label 0")

result = distilBERT(sample_input_difficult, candidate_labels=hogwarts_house)

prediction_string = result['labels'][0]
predicted_label = hogwarts_house.index(prediction_string)
confidence = round(result['scores'][0] * 100, 2)

print(f"Model's Top Guess: Label {predicted_label}")
print(f"Model's Confidence: {confidence}%")
print(f"Reasoning matched to: '{prediction_string}'")


sample_input_easy = "I enjoy teamwork, I work hard and have a stong sense of justice"

print(f"sample input easy: '{sample_input_easy}'")
print(f"actual label should be: Label 1 (hufflepuff)")

result = distilBERT(sample_input_easy, candidate_labels=hogwarts_house)

prediction_string1 = result['labels'][0]
predicted_label1 = hogwarts_house.index(prediction_string1)
confidence1 = round(result['scores'][0] * 100, 2)

print(f"Model's Top Guess: Label {predicted_label1}")
print(f"Model's Confidence: {confidence1}%")
print(f"Reasoning matched to: '{prediction_string1}'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sample input complex: 'I once quit a job on the spot because my boss asked me to sign off on a report which I knew was misleading. Some things are non-negotiable for me'
actual label should be: Label 0
Model's Top Guess: Label 2
Model's Confidence: 31.77%
Reasoning matched to: 'intellectual, curious, analytical, independent thinker'
sample input easy: 'I enjoy teamwork, I work hard and have a stong sense of justice'
actual label should be: Label 1 (hufflepuff)
Model's Top Guess: Label 1
Model's Confidence: 98.96%
Reasoning matched to: 'loyal, hardworking, fair, avoids conflict, team player'


## Run for Training

In [4]:
import numpy as np
from sklearn.metrics import accuracy_score
from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
 )

hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)
print(f"Training on {len(hf_dataset['train'])} examples, validating on {len(hf_dataset['test'])} examples.\n")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_data = hf_dataset.map(tokenize, batched=True)

# referenced AI for distilbert pipeline
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

Training on 160 examples, validating on 40 examples.



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Metric Definition

In [1]:
import inspect
from transformers import EarlyStoppingCallback

def metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Keep shared args separate from version-specific eval strategy.
common_training_args = dict(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.05,
    warmup_ratio=0.1,
    save_strategy='epoch',
    save_total_limit=2,
    metric_for_best_model='eval_loss',
    load_best_model_at_end=True,
    logging_steps=5,
    dataloader_pin_memory=False,
 )

try:
    training_args = TrainingArguments(
        **common_training_args,
        eval_strategy='epoch',
    )
except TypeError:
    training_args = TrainingArguments(
        **common_training_args,
        evaluation_strategy='epoch',
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['test'],
    compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
 )

trainer_init_params = inspect.signature(Trainer.__init__).parameters
if 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer

trainer = Trainer(**trainer_kwargs)

print('training started')
trainer.train()

model_save_path = './hogwarts_classifier_model'
trainer.save_model(model_save_path)
print('Training successful and saved')

KeyboardInterrupt: 

In [ ]:
import json

def clean_notebook_widgets(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets key from metadata
    if 'metadata' in nb and 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']
        print(f"Removed 'widgets' from metadata in {file_path}")
    else:
        print(f"No 'widgets' key found in metadata of {file_path}")

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

# Note: Replace 'your_notebook_name.ipynb' with the actual name if you have downloaded it,
# or use this logic if you are post-processing the file.
# clean_notebook_widgets('your_notebook_name.ipynb')

## Final Inference Tab (Google Colab)
Describe your personality in the form input below, then run the next cell to compare both models.
- Zero-shot model: quick semantic guess
- Fine-tuned model: trained on your custom dataset

In [ ]:
#@title Personality Classifier (Zero-Shot vs Fine-Tuned)
personality_text = "I take charge when things get messy, defend people who are treated unfairly, and I can be impulsive when I care deeply." #@param {type:"string"}

import torch
from transformers import pipeline, DistilBertForSequenceClassification, DistilBertTokenizer

if not personality_text.strip():
    personality_text = input("Describe your personality in 2-5 sentences: ").strip()

# House names used for display
house_names = ["Gryffindor", "Hufflepuff", "Ravenclaw", "Slytherin"]

# Candidate labels for zero-shot model
if "hogwarts_house" not in globals():
    hogwarts_house = [
        "courageous, reckless, principled, confronts authority",
        "loyal, hardworking, fair, avoids conflict, team player",
        "intellectual, curious, analytical, independent thinker",
        "ambitious, strategic, calculating, resourceful, networking",
    ]

# 1) Zero-shot prediction
if "distilBERT" not in globals():
    distilBERT = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")

zs_result = distilBERT(personality_text, candidate_labels=hogwarts_house)
zs_top_label = zs_result["labels"][0]
zs_top_score = float(zs_result["scores"][0])
zs_house_index = hogwarts_house.index(zs_top_label)
zs_house_name = house_names[zs_house_index]

# 2) Fine-tuned model prediction
if "tokenizer" not in globals():
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

if "trainer" in globals() and hasattr(trainer, "model"):
    infer_model = trainer.model
elif "model" in globals():
    infer_model = model
else:
    infer_model = DistilBertForSequenceClassification.from_pretrained('./hogwarts_classifier_model')

infer_model.eval()
inputs = tokenizer(personality_text, return_tensors="pt", truncation=True, padding=True, max_length=128)

with torch.no_grad():
    logits = infer_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().tolist()

ft_index = int(torch.argmax(logits, dim=-1).item())
ft_house_name = house_names[ft_index]
ft_top_score = float(probs[ft_index])

print("Input:")
print(personality_text)
print("\n=== Zero-Shot Model ===")
print(f"Guess: Label {zs_house_index} ({zs_house_name})")
print(f"Certainty: {zs_top_score * 100:.2f}%")
print("Top matches:")
for label_text, score in zip(zs_result["labels"], zs_result["scores"]):
    idx = hogwarts_house.index(label_text)
    print(f"  Label {idx} ({house_names[idx]}): {score * 100:.2f}%")

print("\n=== Fine-Tuned Model ===")
print(f"Guess: Label {ft_index} ({ft_house_name})")
print(f"Certainty: {ft_top_score * 100:.2f}%")
print("Class probabilities:")
for idx, score in enumerate(probs):
    print(f"  Label {idx} ({house_names[idx]}): {score * 100:.2f}%")